# GroupBy

In [29]:
import polars as pl

## Presentación de nuestro Dataset
- `retail_sales.csv` es una lista de transacciones de una tienda de comercio electrónico ficticia.
- Pasemos `try_parse_dates=True` para convertir la columna `purchase_date` a valores de tipo fecha.
- También ordenemos por la columna `purchase_date`.

In [30]:
pl.read_csv("retail_sales.csv", try_parse_dates=True)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,str,str,str,date,str,i64,f64,f64
"""T0000""","""North""","""Electronics""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""T0001""","""West""","""Electronics""","""TV""",2025-05-10,"""Online""",1,113.23,113.23
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5
"""T0003""","""North""","""Toys""","""Puzzle""",2025-04-04,"""Online""",4,319.66,1278.64
"""T0004""","""East""","""Clothing""","""Shirt""",2025-04-08,"""Online""",3,55.59,166.77
…,…,…,…,…,…,…,…,…
"""T0995""","""North""","""Electronics""","""Laptop""",2025-01-17,"""Online""",2,341.58,683.16
"""T0996""","""South""","""Clothing""","""Shirt""",2025-03-25,"""In-Store""",4,258.69,1034.76
"""T0997""","""South""","""Toys""","""Puzzle""",2025-01-28,"""In-Store""",1,120.04,120.04


- Varias columnas tienen un número reducido de valores únicos, por lo que podemos convertirlas a enums.

In [ ]:
pl.read_csv("retail_sales.csv", try_parse_dates=True).select(pl.all().n_unique())

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
u32,u32,u32,u32,u32,u32,u32,u32,u32
1000,4,4,12,181,2,8,989,997


In [ ]:
pl.read_csv("retail_sales.csv", try_parse_dates=True).select(
    pl.col("sales_channel").unique()
)

sales_channel
str
"""Online"""
"""In-Store"""


In [31]:
region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
)

sales.head(3)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0000""","""North""","""Electronics""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""T0001""","""West""","""Electronics""","""TV""",2025-05-10,"""Online""",1,113.23,113.23
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html
- https://docs.pola.rs/api/python/stable/reference/api/polars.datatypes.Enum.html

## Introducción a la Agrupación
- Agrupar simplemente significa "colocar en grupos".
- El valor de agrupar es realizar cálculos agregados sobre subconjuntos de datos.
- La agrupación consta de tres pasos (dividir, aplicar, combinar):
    - **Dividir** los datos en grupos basados en valores distintos de columnas
    - **Aplicar** una función de agregación a cada grupo
    - **Combinar** los resultados en un nuevo `DataFrame`

## El Método group_by
- El método `group_by` acepta la(s) columna(s) cuyos valores únicos determinarán los grupos.
- El `group_by` devuelve un objeto `GroupBy`.
- Polars creará un grupo por cada valor distinto de la columna. Este es el paso de **división**.
- El objeto `GroupBy` es un contenedor para todos los `DataFrames` divididos (un `DataFrame` por cada valor único).

In [32]:
region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
)

sales.head(3)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0000""","""North""","""Electronics""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""T0001""","""West""","""Electronics""","""TV""",2025-05-10,"""Online""",1,113.23,113.23
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5


- El siguiente ejemplo crea grupos usando los valores de la columna `product_category`.
- El objeto `GroupBy` almacenará 4 grupos, uno por cada categoría de producto (`Electronics`, `Clothing`, `Home` y `Toys`).

In [33]:
groups = sales.group_by("product_category")
groups

- El objeto `GroupBy` no es particularmente útil por sí solo, pero soporta muchos métodos para agregación.
- El método `len` devuelve un `DataFrame` con el número de filas por grupo.
- El valor `Clothing` aparece en 236 filas en `sales`, `Electronics` aparece en 230 filas, y así sucesivamente.

In [34]:
groups.len()

product_category,len
enum,u32
"""Electronics""",230
"""Clothing""",236
"""Toys""",268
"""Home""",266


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.group_by.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.len.html

## Métodos de GroupBy
- Invocar un método sobre el objeto `GroupBy` aplica un cálculo a _cada_ grupo y luego combina los resultados en un nuevo `DataFrame`.

In [ ]:
region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
)

groups = sales.group_by("product_category")

- El método `first` devuelve la primera fila de cada grupo.
- Polars colocará la columna de grupo (`product_category`) al inicio del `DataFrame`.
- Observa que invocar el método varias veces devuelve un `DataFrame` diferente cada vez.
- El `DataFrame` **tiene las mismas 4 filas (una por cada categoría de producto) pero aparecen en un orden diferente.**

In [39]:
groups.first()

product_category,transaction_id,region,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
enum,str,enum,str,date,enum,i64,f64,f64
"""Home""","""T0005""","""South""","""Couch""",2025-02-13,"""Online""",1,306.52,306.52
"""Electronics""","""T0000""","""North""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""Clothing""","""T0002""","""West""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5
"""Toys""","""T0003""","""North""","""Puzzle""",2025-04-04,"""Online""",4,319.66,1278.64


- La primera fila de cada grupo corresponde a su aparición en el `DataFrame` original.

- **Polars almacena las _filas_ dentro de cada grupo en el orden en que aparecen en el `DataFrame` original**.
- Por defecto, Polars no mantiene el orden de los _grupos_ en la salida.
- Son los _grupos_ los que aparecen en un orden aparentemente aleatorio cada vez que ejecutamos.
- Internamente, **Polars paraleliza la extracción de una fila de cada grupo**.
- El orden en que se evalúan los resultados puede variar de una ejecución a otra.
- Por lo tanto, **Polars no garantiza consistencia en el _orden_ de las filas. Sin embargo, las filas serán las mismas**.

### El Parámetro `maintain_order`

Cuando estableces el parámetro `maintain_order` en `True`, fuerzas al objeto `GroupBy` a almacenar los grupos en un orden consistente. **Este orden se determina por la primera aparición de los valores distintos en la columna (o columnas) utilizada para la agrupación**. Sin embargo, es importante tener en cuenta que forzar este orden puede ser una desventaja para las operaciones de agregación, ya que requiere que los cálculos se realicen por grupo de manera secuencial, lo que **impide una paralelización efectiva** y, por lo tanto, puede afectar el rendimiento.

En el caso de `product_category`, el orden será `"Electronics"`, `"Clothing"`, `"Toys"` y `"Home"`.

In [ ]:
groups = sales.group_by("product_category", maintain_order=True)

- Ahora el orden de las filas es consistente.

In [ ]:
groups.first()

product_category,transaction_id,region,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
enum,str,enum,str,date,enum,i64,f64,f64
"""Electronics""","""T0000""","""North""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""Clothing""","""T0002""","""West""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5
"""Toys""","""T0003""","""North""","""Puzzle""",2025-04-04,"""Online""",4,319.66,1278.64
"""Home""","""T0005""","""South""","""Couch""",2025-02-13,"""Online""",1,306.52,306.52


- El método `last` devuelve la última fila de cada grupo.

In [ ]:
groups.last()

product_category,transaction_id,region,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
enum,str,enum,str,date,enum,i64,f64,f64
"""Electronics""","""T0995""","""North""","""Laptop""",2025-01-17,"""Online""",2,341.58,683.16
"""Clothing""","""T0996""","""South""","""Shirt""",2025-03-25,"""In-Store""",4,258.69,1034.76
"""Toys""","""T0998""","""North""","""RC Car""",2025-02-24,"""In-Store""",3,10.93,32.79
"""Home""","""T0999""","""East""","""Vacuum""",2025-04-16,"""In-Store""",1,443.3,443.3


- El método `head` devuelve un número especificado de filas desde el inicio de cada grupo.

In [ ]:
groups.head(2)

product_category,transaction_id,region,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
enum,str,enum,str,date,enum,i64,f64,f64
"""Electronics""","""T0000""","""North""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""Electronics""","""T0001""","""West""","""TV""",2025-05-10,"""Online""",1,113.23,113.23
"""Clothing""","""T0002""","""West""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5
"""Clothing""","""T0004""","""East""","""Shirt""",2025-04-08,"""Online""",3,55.59,166.77
"""Toys""","""T0003""","""North""","""Puzzle""",2025-04-04,"""Online""",4,319.66,1278.64
"""Toys""","""T0009""","""East""","""Puzzle""",2025-05-26,"""In-Store""",1,109.44,109.44
"""Home""","""T0005""","""South""","""Couch""",2025-02-13,"""Online""",1,306.52,306.52
"""Home""","""T0008""","""West""","""Couch""",2025-03-25,"""Online""",5,113.71,568.55


- El método `tail` devuelve un número especificado de filas desde el final de cada grupo.

In [ ]:
groups.tail(2)

product_category,transaction_id,region,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
enum,str,enum,str,date,enum,i64,f64,f64
"""Electronics""","""T0993""","""North""","""Laptop""",2025-06-03,"""Online""",3,224.77,674.31
"""Electronics""","""T0995""","""North""","""Laptop""",2025-01-17,"""Online""",2,341.58,683.16
"""Clothing""","""T0994""","""West""","""Shirt""",2025-02-21,"""Online""",3,454.83,1364.49
"""Clothing""","""T0996""","""South""","""Shirt""",2025-03-25,"""In-Store""",4,258.69,1034.76
"""Toys""","""T0997""","""South""","""Puzzle""",2025-01-28,"""In-Store""",1,120.04,120.04
"""Toys""","""T0998""","""North""","""RC Car""",2025-02-24,"""In-Store""",3,10.93,32.79
"""Home""","""T0992""","""North""","""Vacuum""",2025-02-25,"""Online""",4,182.25,729.0
"""Home""","""T0999""","""East""","""Vacuum""",2025-04-16,"""In-Store""",1,443.3,443.3


### Lectura Adicional
- https://docs.pola.rs/user-guide/getting-started/#group_by
- https://docs.pola.rs/user-guide/concepts/expressions-and-contexts/#group_by-and-aggregations
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.first.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.last.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.head.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.tail.html

## Valores Más Grandes y Más Pequeños por Grupo

In [40]:
region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
)

groups = sales.group_by("product_category", maintain_order=True)

sales.head(3)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0000""","""North""","""Electronics""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""T0001""","""West""","""Electronics""","""TV""",2025-05-10,"""Online""",1,113.23,113.23
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5


- El método `max` devuelve el valor más grande _por columna_ para cada grupo.
- **Estas _no_ son filas individuales del `DataFrame` `sales` original**.
- Para la categoría de producto `"Toys"`, `"T0998"` es el mayor `transaction_id`, 10 es la mayor `quantity`, y así sucesivamente.

In [41]:
groups.max()

product_category,transaction_id,region,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
enum,str,enum,str,date,enum,i64,f64,f64
"""Electronics""","""T0995""","""South""","""TV""",2025-06-30,"""Online""",6,497.98,2487.85
"""Clothing""","""T0996""","""South""","""Shirt""",2025-06-30,"""Online""",5,498.0,2437.0
"""Toys""","""T0998""","""South""","""RC Car""",2025-06-30,"""Online""",10,498.68,2470.05
"""Home""","""T0999""","""South""","""Vacuum""",2025-06-30,"""Online""",8,496.38,2478.85


- Veamos la fila con un `transaction_id` de 999. **Observa que los valores no coinciden con la fila anterior**.

In [42]:
sales.filter(pl.col("transaction_id") == "T0999")

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0999""","""East""","""Home""","""Vacuum""",2025-04-16,"""In-Store""",1,443.3,443.3


- El método `min` devuelve el valor más pequeño por columna para cada grupo.

In [43]:
groups.min()

product_category,transaction_id,region,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
enum,str,enum,str,date,enum,i64,f64,f64
"""Electronics""","""T0000""","""East""","""Headphones""",2025-01-02,"""In-Store""",1,6.75,11.59
"""Clothing""","""T0002""","""East""","""Jacket""",2025-01-02,"""In-Store""",1,8.89,19.06
"""Toys""","""T0003""","""East""","""Doll""",2025-01-01,"""In-Store""",1,6.61,15.12
"""Home""","""T0005""","""East""","""Couch""",2025-01-01,"""In-Store""",1,5.27,10.54


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.max.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.min.html

## Agregaciones con el Método `agg`
- Los métodos de la lección anterior realizaban cálculos de agregación en _todas_ las columnas de cada grupo.
- El método `agg` se enfoca en columna(s) específica(s) para los cálculos de grupo.

In [44]:
region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
)

groups = sales.group_by("product_category", maintain_order=True)

sales.head(3)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0000""","""North""","""Electronics""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""T0001""","""West""","""Electronics""","""TV""",2025-05-10,"""Online""",1,113.23,113.23
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5


- El método `agg` acepta una o más expresiones.
- Calculemos el valor más grande en la columna `quantity` para cada categoría de producto.
- Los resultados son los mismos que invocar `groups.max` sin todas las columnas extra.

In [ ]:
sales.head(2)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0000""","""North""","""Electronics""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""T0001""","""West""","""Electronics""","""TV""",2025-05-10,"""Online""",1,113.23,113.23


- ¡Se aceptan múltiples expresiones! Como siempre, todos los nombres de columna en el nuevo `DataFrame` deben ser únicos.

In [45]:
groups.agg(
    pl.col("quantity").max().alias("largest_quantity"),
    pl.col("quantity").min().alias("smallest_quantity"),
    pl.col("quantity").mean().alias("average_quantity"),
    pl.col("quantity").median().alias("median_quantity"),
)

product_category,largest_quantity,smallest_quantity,average_quantity,median_quantity
enum,i64,i64,f64,f64
"""Electronics""",6,1,2.943478,3.0
"""Clothing""",5,1,3.004237,3.0
"""Toys""",10,1,3.130597,3.0
"""Home""",8,1,2.988722,3.0


- El método `sum` suma los valores en la columna especificada para cada grupo.

In [48]:
groups.agg(pl.col("total_price").sum().alias("total_spend_per_category"))

product_category,total_spend_per_category
enum,f64
"""Electronics""",168243.25
"""Clothing""",172431.2
"""Toys""",212250.02
"""Home""",193942.34


In [47]:
sales.filter(pl.col("product_category") == "Toys").select(pl.col("total_price").sum())

total_price
f64
212250.02


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/aggregation/#basic-aggregations
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.agg.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.max.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.min.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.mean.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.median.html

## Uso de Selectores en Agregaciones
- Usar el método `agg` para enfocarse en columnas individuales para agregaciones está bien, pero puede volverse verboso.
- El submódulo `selectors` puede ser útil para seleccionar todas las columnas de un tipo de dato específico.

El uso de `cs.numeric()` (del módulo polars.selectors) es muy útil cuando trabajas con `group_by` y `agg` por tres razones principales:

- **Automatización:** En lugar de escribir manualmente cada columna (`pl.col('quantity')`, `pl.col('unit_price')`, etc.), el selector identifica automáticamente todas las columnas de tipo entero o flotante.
- **Prevención de Errores:** Si intentas aplicar una función matemática como `.sum()` o `.mean()` a una columna de texto (como `product_id`), Polars lanzará un error. `cs.numeric()` garantiza que la operación solo se aplique a columnas donde el cálculo tiene sentido.
- **Escalabilidad:** Si tu dataset cambia y se añaden nuevas columnas numéricas en el futuro, el código seguirá funcionando sin necesidad de actualizar la lista de columnas en el método `agg`.

In [49]:
import polars.selectors as cs

In [50]:
region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
)

groups = sales.group_by("product_category", maintain_order=True)

sales.head(3)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0000""","""North""","""Electronics""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""T0001""","""West""","""Electronics""","""TV""",2025-05-10,"""Online""",1,113.23,113.23
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5


- Calculemos la suma de los valores en cada columna numérica para cada grupo.
- Usaremos `cs.numeric` para crear un selector que apunte a todas las columnas numéricas (enteros y flotantes).
- Luego calcularemos la suma de los valores por categoría de producto.

In [51]:
groups.agg(cs.numeric().sum())

product_category,quantity,unit_price,total_price
enum,i64,f64,f64
"""Electronics""",677,56478.13,168243.25
"""Clothing""",709,58964.64,172431.2
"""Toys""",839,67275.65,212250.02
"""Home""",795,66723.32,193942.34


- El método `name.suffix` concatena una cadena consistente al final de cada nombre de columna.
- Ofrece un atajo para evitar nombrar cada columna individualmente.

In [52]:
groups.agg(cs.numeric().sum().name.suffix("_sum"))

product_category,quantity_sum,unit_price_sum,total_price_sum
enum,i64,f64,f64
"""Electronics""",677,56478.13,168243.25
"""Clothing""",709,58964.64,172431.2
"""Toys""",839,67275.65,212250.02
"""Home""",795,66723.32,193942.34


- Las funciones de agregación como `sum` o `mean` se evalúan como expresiones de Polars.
- Podemos combinar las expresiones pasadas al método `agg`.
- El siguiente ejemplo calcula la suma y el promedio de los valores en todas las columnas numéricas.

In [53]:
groups.agg(
    cs.numeric().sum().name.suffix("_sum"), cs.numeric().mean().name.suffix("_average")
)

product_category,quantity_sum,unit_price_sum,total_price_sum,quantity_average,unit_price_average,total_price_average
enum,i64,f64,f64,f64,f64,f64
"""Electronics""",677,56478.13,168243.25,2.943478,245.557087,731.492391
"""Clothing""",709,58964.64,172431.2,3.004237,249.850169,730.640678
"""Toys""",839,67275.65,212250.02,3.130597,251.028545,791.977687
"""Home""",795,66723.32,193942.34,2.988722,250.839549,729.106541


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/selectors.html#polars.selectors.numeric
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.name.suffix.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.mean.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.sum.html

## Agrupación con Múltiples Columnas
- Pasa una lista de valores a `group_by` para agrupar por múltiples valores de columna.
- **Usa la columna con menor número de valores únicos como el grupo exterior/primero**.

In [55]:
region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
)

groups = sales.group_by(["sales_channel", "region"], maintain_order=True)

sales.head(3)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0000""","""North""","""Electronics""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""T0001""","""West""","""Electronics""","""TV""",2025-05-10,"""Online""",1,113.23,113.23
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5


In [56]:
groups.agg(
    pl.col("total_price").mean().alias("average_spend_per_region_and_sales_channel"),
    pl.col("quantity").sum().alias("total_sum_per_region_and_sales_channel"),
)

sales_channel,region,average_spend_per_region_and_sales_channel,total_sum_per_region_and_sales_channel
enum,enum,f64,i64
"""Online""","""North""",770.837029,402
"""Online""","""West""",717.891985,395
"""Online""","""East""",738.563772,363
"""Online""","""South""",764.381031,313
"""In-Store""","""East""",710.9988,346
"""In-Store""","""West""",754.072615,402
"""In-Store""","""South""",799.673917,365
"""In-Store""","""North""",725.800414,434


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/aggregation/#nested-grouping
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.group_by.html

## Agrupación de Datos Temporales
- El método `group_by` usa valores distintos para definir los grupos.
- **Agrupar valores de fecha y hora por tiempo exacto generalmente no es útil debido a la gran cantidad de variación**.
- Una ventana de tiempo es un intervalo de tiempo (por ejemplo, cada 1 semana).
- El método `group_by_dynamic` crea grupos usando una ventana de tiempo.
- El método `group_by_dynamic` busca la inclusión de un valor de fecha y hora dentro de la ventana/rango de tiempo.
- Por ejemplo, una ventana de tiempo del 1 al 7 de enero incluirá una fila cuya fecha sea el 5 de enero.

In [57]:
region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
).sort("purchase_date")  # Importante!

sales.head(3)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0763""","""East""","""Toys""","""RC Car""",2025-01-01,"""Online""",4,116.14,464.56
"""T0931""","""North""","""Home""","""Couch""",2025-01-01,"""In-Store""",5,442.14,2210.7
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5


- **Asegúrate de _ordenar_ por la columna de fecha/datetime cuyos valores se agruparán.**
- El parámetro `every` acepta un intervalo de tiempo.
- El siguiente ejemplo proporciona `1w` para 1 semana, pero podemos usar símbolos para designar otras duraciones.
- **Por defecto, Polars encuentra la fecha más temprana y la agrupa en una ventana semanal que comienza el lunes.**
- La fecha más temprana en `sales` es 01/01/2025, un miércoles.
- Por lo tanto, el primer grupo comienza desde el lunes de esa semana, 30/12/2024.


In [58]:
groups = sales.group_by_dynamic("purchase_date", every="1w")

- Calculemos la suma de artículos vendidos para cada agrupación de 1 semana.

In [59]:
groups.agg(pl.col("total_price").sum())

purchase_date,total_price
date,f64
2024-12-30,15719.96
2025-01-06,26694.13
2025-01-13,21710.04
2025-01-20,25602.3
2025-01-27,32929.04
…,…
2025-06-02,34910.79
2025-06-09,38389.9
2025-06-16,30332.83


- Polars luego crea el rango de semanas hasta llegar a la última fecha del dataset.
- El argumento `start_by` acepta una cadena con un día de la semana especificado.
- El siguiente ejemplo le pide a Polars que inicie una semana/rango los miércoles.
- Cada grupo/intervalo ahora va de miércoles a martes.

In [60]:
sales.group_by_dynamic("purchase_date", every="1w", start_by="wednesday").agg(
    pl.col("total_price").sum()
)

purchase_date,total_price
date,f64
2025-01-01,21703.82
2025-01-08,27109.7
2025-01-15,22062.4
2025-01-22,30232.26
2025-01-29,33712.96
…,…
2025-05-28,29783.05
2025-06-04,41570.55
2025-06-11,28358.12


- Alternativamente, podemos pasar al parámetro `start_by` el valor `datapoint`.
- Polars creará intervalos de 1 semana comenzando desde el primer punto de datos, `2025-01-01`.

In [61]:
sales.group_by_dynamic("purchase_date", every="1w", start_by="datapoint").agg(
    pl.col("total_price").sum()
)

purchase_date,total_price
date,f64
2025-01-01,21703.82
2025-01-08,27109.7
2025-01-15,22062.4
2025-01-22,30232.26
2025-01-29,33712.96
…,…
2025-05-28,29783.05
2025-06-04,41570.55
2025-06-11,28358.12


- El símbolo `mo` representa mes.
- El siguiente ejemplo crea agrupaciones mensuales.
- Polars es lo suficientemente inteligente como para tener en cuenta las diferencias en la duración de los meses.

In [62]:
sales.group_by_dynamic("purchase_date", every="2mo", start_by="datapoint").agg(
    pl.col("total_price").sum()
)

purchase_date,total_price
date,f64
2025-01-01,232185.19
2025-03-01,240708.59
2025-05-01,273973.03


- El siguiente ejemplo agrupa por trimestre.

In [63]:
sales.group_by_dynamic("purchase_date", every="3mo", start_by="datapoint").agg(
    pl.col("total_price").sum()
)

purchase_date,total_price
date,f64
2025-01-01,358463.96
2025-04-01,388402.85


- Podemos combinar diferentes especificaciones de tiempo.
- El siguiente ejemplo crea grupos separados por 1 mes, 2 semanas y 3 días.

In [64]:
sales.group_by_dynamic("purchase_date", every="1mo2w3d", start_by="datapoint").agg(
    pl.col("total_price").sum()
)

purchase_date,total_price
date,f64
2025-01-01,193152.24
2025-02-18,175349.67
2025-04-04,184713.01
2025-05-21,193651.89


- Existen opciones adicionales de configuración de ventanas de tiempo:
    - sin superposición, donde cada grupo de período no se superpone con otro
    - con superposición (los grupos de período pueden superponerse entre sí)
    - con espacios (sin superposición, con intervalos entre períodos consecutivos)

### Lectura Adicional
- https://docs.pola.rs/user-guide/transformations/time-series/rolling/#parameters-for-group_by_dynamic
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.group_by_dynamic.html
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.dataframe.group_by.GroupBy.agg.html

## Funciones de Ventana y el Método `over`

El método `over` en Polars es fundamental para trabajar con funciones de ventana. Aquí tienes una explicación detallada de su utilidad:

**Cálculos por Grupo sin Agregar:** A diferencia de `group_by().agg()`, que reduce el número de filas (colapsa los datos), el método over realiza cálculos sobre grupos pero **mantiene el número original de filas** en el DataFrame. Esto permite, por ejemplo, comparar cada fila individual con el promedio de su grupo.

**Definición de Ventana:** Cuando usas **.over("columna")**, le estás diciendo a Polars: "mira solo el subconjunto de filas que comparten el mismo valor en esta columna para hacer el cálculo".

**Uso Común:** Se utiliza frecuentemente con funciones como `rank()`, `mean()`, `sum()`, `max()` o `min()` para crear nuevas columnas que contengan métricas contextuales de grupo al lado de los datos originales.



In [65]:
region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
)

sales.head(3)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0000""","""North""","""Electronics""","""Headphones""",2025-02-05,"""Online""",3,126.22,378.66
"""T0001""","""West""","""Electronics""","""TV""",2025-05-10,"""Online""",1,113.23,113.23
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5


El método `rank()` en Polars se utiliza para asignar un orden numérico (posición) en base a los valores de una columna.

**Propósito:** Determina la posición de cada elemento respecto a los demás. Por ejemplo, en una columna de precios, puede decirte cuál es el 1º más caro, el 2º, etc.

**Dirección:**
- `descending=True`: El valor más alto recibe el rango 1 (útil para encontrar los 'top' ventas).
- `descending=False` (por defecto): El valor más bajo recibe el rango 1.

**Contexto Global vs. Grupos:**
- Si se usa solo, calcula el rango sobre todo el DataFrame.
- Si se combina con `.over("columna")`, se convierte en una función de ventana. Esto permite calcular rangos dentro de subgrupos (por ejemplo, ver cuál es el producto más vendido en cada región por separado).

**Tipos de datos:** Generalmente devuelve valores flotantes, por lo que a menudo se usa `.cast(pl.UInt32)` para convertir el rango a un número entero limpio.

In [67]:
sales.with_columns(
    pl.col("total_price")
      .rank(descending=True)
      .cast(pl.UInt32)
      .alias("price_rank")
).sort("price_rank")

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price,price_rank
str,enum,enum,str,date,enum,i64,f64,f64,u32
"""T0189""","""West""","""Electronics""","""Laptop""",2025-02-16,"""In-Store""",5,497.57,2487.85,1
"""T0586""","""West""","""Home""","""Lamp""",2025-03-25,"""In-Store""",5,495.77,2478.85,2
"""T0055""","""South""","""Toys""","""RC Car""",2025-06-02,"""In-Store""",5,494.01,2470.05,3
"""T0142""","""East""","""Clothing""","""Jeans""",2025-06-09,"""Online""",5,487.4,2437.0,4
"""T0608""","""North""","""Clothing""","""Shirt""",2025-05-24,"""Online""",5,483.56,2417.8,5
…,…,…,…,…,…,…,…,…,…
"""T0734""","""East""","""Home""","""Vacuum""",2025-06-21,"""In-Store""",1,12.89,12.89,996
"""T0340""","""North""","""Electronics""","""Headphones""",2025-02-25,"""Online""",1,11.83,11.83,997
"""T0862""","""North""","""Electronics""","""Laptop""",2025-06-17,"""In-Store""",1,11.59,11.59,998


- Supongamos que queremos clasificar las transacciones _dentro_ de cada región.
- Queremos que las transacciones más rentables en las regiones `"West"`, `"North"`, `"South"` y `"East"` tengan un rango de 1.
- El método `over` especifica que el ranking debe hacerse _por separado_ para cada grupo.
- Le pedimos a Polars que clasifique los valores numéricos en `total_price` sobre los grupos definidos por `region`.
- Se llama "función de ventana" porque cada cálculo opera sobre una ventana -- un subconjunto/grupo de filas -- en lugar de todo el dataset.


In [ ]:
sales.with_columns(
    pl.col("total_price")
    .rank(descending=True)
    .over("region")
    .cast(pl.UInt32)
    .alias("rank")
).sort(["rank", "region"])

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price,rank
str,enum,enum,str,date,enum,i64,f64,f64,u32
"""T0142""","""East""","""Clothing""","""Jeans""",2025-06-09,"""Online""",5,487.4,2437.0,1
"""T0189""","""West""","""Electronics""","""Laptop""",2025-02-16,"""In-Store""",5,497.57,2487.85,1
"""T0608""","""North""","""Clothing""","""Shirt""",2025-05-24,"""Online""",5,483.56,2417.8,1
"""T0055""","""South""","""Toys""","""RC Car""",2025-06-02,"""In-Store""",5,494.01,2470.05,1
"""T0474""","""East""","""Toys""","""Doll""",2025-03-04,"""Online""",5,459.71,2298.55,2
…,…,…,…,…,…,…,…,…,…
"""T0998""","""North""","""Toys""","""RC Car""",2025-02-24,"""In-Store""",3,10.93,32.79,279
"""T0719""","""North""","""Home""","""Lamp""",2025-03-02,"""Online""",4,7.26,29.04,280
"""T0304""","""North""","""Home""","""Vacuum""",2025-03-29,"""Online""",1,16.18,16.18,281


- Podemos pasar múltiples columnas al método `over`.
- La ventana aquí abarca cada combinación de `sales_channel` y `region`.
- Hay 8 combinaciones posibles de `sales_channel` (2 opciones) y `region` (4 opciones).
- Como resultado, 8 filas tendrán un rango de 1, 8 filas tendrán un rango de 2, y así sucesivamente.

In [ ]:
sales.with_columns(
    pl.col("total_price")
    .rank(descending=True)
    .over("region", "sales_channel")
    .cast(pl.UInt32)
    .alias("rank")
).sort("rank", "sales_channel")

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price,rank
str,enum,enum,str,date,enum,i64,f64,f64,u32
"""T0624""","""East""","""Electronics""","""Headphones""",2025-04-29,"""In-Store""",5,452.32,2261.6,1
"""T0055""","""South""","""Toys""","""RC Car""",2025-06-02,"""In-Store""",5,494.01,2470.05,1
"""T0457""","""North""","""Toys""","""Puzzle""",2025-02-04,"""In-Store""",5,478.31,2391.55,1
"""T0189""","""West""","""Electronics""","""Laptop""",2025-02-16,"""In-Store""",5,497.57,2487.85,1
"""T0608""","""North""","""Clothing""","""Shirt""",2025-05-24,"""Online""",5,483.56,2417.8,1
…,…,…,…,…,…,…,…,…,…
"""T0583""","""North""","""Clothing""","""Jacket""",2025-03-01,"""In-Store""",1,55.11,55.11,141
"""T0821""","""North""","""Clothing""","""Jeans""",2025-04-11,"""In-Store""",4,8.89,35.56,142
"""T0133""","""North""","""Home""","""Lamp""",2025-05-03,"""In-Store""",1,35.11,35.11,143


### Lectura Adicional
- https://docs.pola.rs/user-guide/expressions/window-functions/#operations-per-group
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.rank.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.over.html
- https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.cast.html

#Ejercicios de Práctica

In [ ]:
import polars as pl
import polars.selectors as cs

region_enum = pl.Enum(["East", "West", "North", "South"])
product_category_enum = pl.Enum(["Home", "Electronics", "Toys", "Clothing"])
sales_channel_enum = pl.Enum(["In-Store", "Online"])

sales = pl.read_csv(
    "retail_sales.csv",
    try_parse_dates=True,
    schema_overrides={
        "region": region_enum,
        "product_category": product_category_enum,
        "sales_channel": sales_channel_enum,
    },
).sort("purchase_date")

sales.head(3)

transaction_id,region,product_category,product_id,purchase_date,sales_channel,quantity,unit_price,total_price
str,enum,enum,str,date,enum,i64,f64,f64
"""T0763""","""East""","""Toys""","""RC Car""",2025-01-01,"""Online""",4,116.14,464.56
"""T0931""","""North""","""Home""","""Couch""",2025-01-01,"""In-Store""",5,442.14,2210.7
"""T0002""","""West""","""Clothing""","""Jeans""",2025-01-02,"""Online""",5,142.7,713.5


### Ejercicio 1
Agrupa el `DataFrame` `sales` por la columna `region` y calcula el número de transacciones (filas) en cada región.

### Ejercicio 2
Usando el método `agg`, agrupa por `region` y calcula:
- La suma de `total_price` (alias: `"ingresos_totales"`)
- El promedio de `quantity` (alias: `"cantidad_promedio"`)
- El número total de filas por grupo (alias: `"num_transacciones"`)

### Ejercicio 3
Agrupa por `sales_channel` y `product_category` (múltiples columnas) y calcula la suma de `total_price` para cada combinación. ¿Cuál combinación de canal de venta y categoría genera más ingresos?

### Ejercicio 4
Usa `group_by_dynamic` para agrupar las ventas por mes (`"1mo"`) basándote en la columna `purchase_date` y comenzando desde la fecha con el primer dato. Para cada mes, calcula la suma de `quantity` y la suma de `total_price`.

### Ejercicio 5
Usando una función de ventana con `over`, agrega una columna llamada `"rank_por_region"` que clasifique cada transacción por `total_price` en orden descendente _dentro de cada región_. Luego filtra para mostrar solo las 3 transacciones con mayor `total_price` de cada región